# Phase 2 - Spark Reads Original ARCO-ERA5 Data from MinIO

This notebook reads the original January 2024 weather partition directly with Spark S3A. It replaces the earlier approach that loaded small JSONL objects with Boto3 and Pandas before creating a Spark DataFrame.

Production counterpart: `spark_jobs/validate_raw_weather.py`.

In [1]:
import sys
from pathlib import Path
from pyspark.sql import functions as F

PROJECT_ROOT = Path("/workspace")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from spark_jobs.common import create_spark_session, load_settings, normalized_timestamp, s3a_uri
from spark_jobs.validate_raw_weather import REQUIRED_COLUMNS

YEAR, MONTH = 2024, 1
settings = load_settings()
spark = create_spark_session("phase2-read-original-arco-era5", settings)
source = s3a_uri(settings.raw_bucket, f"arco_era5_us_airport_hourly/year={YEAR}/month={MONTH:02d}")
print("Reading:", source)

Reading: s3a://raw/arco_era5_us_airport_hourly/year=2024/month=01


In [2]:
raw_weather_df = spark.read.parquet(source)
missing = sorted(set(REQUIRED_COLUMNS) - set(raw_weather_df.columns))
assert not missing, f"Missing original weather columns: {missing}"
raw_weather_df.printSchema()

root
 |-- time_utc: long (nullable = true)
 |-- airport_key: integer (nullable = true)
 |-- day: short (nullable = true)
 |-- hour_utc: byte (nullable = true)
 |-- 2m_dewpoint_temperature: float (nullable = true)
 |-- 2m_temperature: float (nullable = true)
 |-- 10m_u_component_of_wind: float (nullable = true)
 |-- 10m_v_component_of_wind: float (nullable = true)
 |-- surface_pressure: float (nullable = true)
 |-- 10m_wind_gust_since_previous_post_processing: float (nullable = true)
 |-- mean_sea_level_pressure: float (nullable = true)
 |-- total_precipitation: float (nullable = true)
 |-- total_cloud_cover: float (nullable = true)
 |-- convective_available_potential_energy: float (nullable = true)
 |-- airport_shard: integer (nullable = true)



In [3]:
validated_weather_df = raw_weather_df.withColumn(
    "normalized_time_utc", normalized_timestamp(raw_weather_df, "time_utc")
)
summary = validated_weather_df.agg(
    F.count("*").alias("row_count"),
    F.countDistinct("airport_key").alias("airport_count"),
    F.min("normalized_time_utc").alias("min_time_utc"),
    F.max("normalized_time_utc").alias("max_time_utc"),
)
summary.show(truncate=False)
validated_weather_df.select("normalized_time_utc", "airport_key", "2m_temperature", "total_precipitation").show(10, truncate=False)

+---------+-------------+-------------------+-------------------+
|row_count|airport_count|min_time_utc       |max_time_utc       |
+---------+-------------+-------------------+-------------------+
|18693744 |25126        |2024-01-01 00:00:00|2024-01-31 23:00:00|
+---------+-------------+-------------------+-------------------+



+-------------------+-----------+--------------+-------------------+
|normalized_time_utc|airport_key|2m_temperature|total_precipitation|
+-------------------+-----------+--------------+-------------------+
|2024-01-01 00:00:00|10000      |287.07742     |1.3416633E-5       |
|2024-01-01 00:00:00|10001      |289.91824     |0.0                |
|2024-01-01 00:00:00|10002      |285.5811      |8.381903E-7        |
|2024-01-01 00:00:00|10003      |286.20532     |1.9452721E-4       |
|2024-01-01 00:00:00|10004      |291.0304      |0.0                |
|2024-01-01 00:00:00|10005      |287.46622     |9.474903E-5        |
|2024-01-01 00:00:00|10006      |287.5545      |3.353879E-5        |
|2024-01-01 00:00:00|10007      |289.7587      |0.0                |
|2024-01-01 00:00:00|10008      |285.925       |1.4757179E-4       |
|2024-01-01 00:00:00|10009      |274.5663      |0.0                |
+-------------------+-----------+--------------+-------------------+
only showing top 10 rows


## Verified January 2024 Result

The validated partition contains **18,693,744 original airport-hour rows** for **25,126 airport facilities**, spanning `2024-01-01 00:00:00` through `2024-01-31 23:00:00` UTC.

In [4]:
spark.stop()